# Milestone 1 — Reproduce the frame-semantic-transformer baseline

Confirms we reproduce the upstream **`base`** model's published FrameNet 1.7
test-set F1 (trigger `0.74`, frame `0.89`, args `0.75`) with our own eval
harness, so later "we beat it" comparisons are credible.

**Robustness notes (why this version doesn't hit the earlier errors):**
- The parser is **cloned and put on `sys.path`** rather than `pip install`ed.
  There is no package build step, so the `KeyError: 'frame_semantic_transformer'`
  import failure cannot happen.
- We **do not touch** Colab's `protobuf`, `torch`, or `numpy` (pinning them is
  what caused the conflicts). The protobuf error is defeated with an env var
  set *before* any import — see Cell 4.

> **Before running:** `Runtime → Change runtime type → GPU` (A100 / L4 ideal),
> then `Runtime → Run all`. No restart should be needed on a fresh runtime.

In [ ]:
!nvidia-smi

## 1. Get the source + install runtime deps

Clone (no build) then install only what's missing. Pins mirror
`requirements-colab.txt`. `torch` / `protobuf` / `numpy` are left as Colab ships
them on purpose.

In [ ]:
# Clone the vendored parser at the exact commit in PROVENANCE.md (no pip build).
![ -d frame-semantic-transformer ] || git clone -q https://github.com/texturejc/frame-semantic-transformer.git
!cd frame-semantic-transformer && git checkout -q 18cb3023bbb6df0c1b53a52182135c0c0132c073 && echo "checked out $(git rev-parse --short HEAD)"
# Runtime deps only. NOTE: no torch, no protobuf, no numpy here (see markdown above).
!pip install -q "transformers>=4.38,<4.45" "tokenizers>=0.15,<0.20" "sentencepiece>=0.1.99" "nltk>=3.8"

## 2. Imports

The protobuf env var **must** be set before importing transformers/sentencepiece;
that is what prevents the *"Descriptors cannot be created directly"* protobuf
error without downgrading anything. `sys.path.insert` makes the cloned source
importable directly — no install required.

In [ ]:
import os
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"  # <-- before transformers import

import sys, time
sys.path.insert(0, "/content/frame-semantic-transformer")
from collections import defaultdict

import torch
from torch.utils.data import DataLoader
from transformers import T5ForConditionalGeneration, T5TokenizerFast

from frame_semantic_transformer.constants import MODEL_MAX_LENGTH, MODEL_REVISION
from frame_semantic_transformer.data.LoaderDataCache import LoaderDataCache
from frame_semantic_transformer.data.TaskSampleDataset import TaskSampleDataset
from frame_semantic_transformer.data.loaders.framenet17 import (
    Framenet17InferenceLoader, Framenet17TrainingLoader,
)
from frame_semantic_transformer.data.tasks_from_annotated_sentences import (
    tasks_from_annotated_sentences,
)
from frame_semantic_transformer.training.evaluate_batch import (
    TaskEvalResults, calc_eval_metrics, evaluate_batch,
)
from frame_semantic_transformer.training.TrainingModelWrapper import merge_metrics
print("imports OK — frame_semantic_transformer loaded from source")

In [ ]:
import nltk
for pkg in ["framenet_v17", "wordnet", "omw-1.4"]:
    nltk.download(pkg)

## 3. Load the model + build the test dataset

The upstream `base` model lives on the HF hub at revision `v0.2.0`.

In [ ]:
MODEL = "base"      # or "small"
SPLIT = "test"      # or "dev"
BATCH_SIZE = 16

model_ref = f"chanind/frame-semantic-transformer-{MODEL}"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = T5ForConditionalGeneration.from_pretrained(model_ref, revision=MODEL_REVISION).to(device)
tokenizer = T5TokenizerFast.from_pretrained(
    model_ref, revision=MODEL_REVISION, model_max_length=MODEL_MAX_LENGTH, legacy=False
)
print("loaded", model_ref, "on", device)

inference_loader = Framenet17InferenceLoader(); inference_loader.setup()
loader_cache = LoaderDataCache(inference_loader)

training_loader = Framenet17TrainingLoader(); training_loader.setup()
sentences = (training_loader.load_test_data() if SPLIT == "test"
             else training_loader.load_validation_data())
tasks = tasks_from_annotated_sentences(sentences, loader_cache)
dataset = TaskSampleDataset(tasks, tokenizer, balance_tasks=False)
print(f"{len(sentences)} sentences -> {len(tasks)} tasks -> {len(dataset)} samples")

## 4. Evaluate

Manual loop = upstream `TrainingModelWrapper.test_step` without PyTorch-Lightning
(whose 1.x API was removed in 2.0). Scoring functions are upstream's, verbatim.

In [ ]:
loader = DataLoader(dataset, batch_size=BATCH_SIZE, num_workers=2)
merged = defaultdict(TaskEvalResults)
model.eval()
t0 = time.time()
with torch.no_grad():
    for i, batch in enumerate(loader):
        merged = merge_metrics([merged, evaluate_batch(model, tokenizer, batch, loader_cache)])
        if (i + 1) % 10 == 0 or (i + 1) == len(loader):
            print(f"batch {i + 1}/{len(loader)}", flush=True)
elapsed = time.time() - t0

In [ ]:
REPORTED = {"test": {"trigger_identification": 0.74, "frame_classification": 0.89, "args_extraction": 0.75},
            "dev":  {"trigger_identification": 0.78, "frame_classification": 0.91, "args_extraction": 0.78}}[SPLIT]

print(f"{'task':<24}{'precision':>10}{'recall':>10}{'f1':>10}{'reported':>10}")
print("-" * 64)
for task in ("trigger_identification", "frame_classification", "args_extraction"):
    if task not in merged:
        continue
    s = calc_eval_metrics(merged[task].scores)
    print(f"{task:<24}{s['precision']:>10.3f}{s['recall']:>10.3f}{s['f_score']:>10.3f}{REPORTED[task]:>10.2f}")
print("-" * 64)
print(f"{len(dataset)} samples in {elapsed:.1f}s ({1000*elapsed/max(len(dataset),1):.1f} ms/sample)")

## Interpreting

F1 within **~1 point** of the reported column = baseline reproduced ✅. Paste the
table back and it gets recorded in `MILESTONES.md`. The **ms/sample** figure is
the speed target the encoder model (Milestone 2) must beat.

### If something still breaks
- **Any protobuf / "Descriptors" error** → you likely ran an `import transformers`
  *before* Cell 4 set the env var. `Runtime → Restart session`, then `Run all`.
- **CUDA / torch error** → you (or a stray `pip install torch`) replaced Colab's
  torch. `Runtime → Disconnect and delete runtime`, reconnect, `Run all` — do not
  install torch.
- **`frame_semantic_transformer` not found** → the clone cell didn't run; run
  Cell 3 (the git clone), then Cell 4.